### Bước 1: Khởi tạo môi trường
Chúng ta tiến hành cài đặt các thư viện cần thiết bao gồm thư viện kết nối BigQuery, xử lý dữ liệu Pandas và công cụ biến đổi dữ liệu DBT (`dbt-bigquery`).

In [35]:
%pip install -q google-cloud-bigquery pandas pyarrow dbt-bigquery

### Bước 2: Xác thực tài khoản Google Cloud và kết nối BigQuery
Thực hiện xác thực quyền truy cập tài khoản để kết nối trực tiếp đến dự án Google Cloud Platform và thiết lập BigQuery Client.

In [36]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import os

auth.authenticate_user()

PROJECT_ID = "formidable-app-510501-p3"

client = bigquery.Client(project=PROJECT_ID)

print("Đã kết nối project:", client.project)

Đã kết nối project: formidable-app-510501-p3


### Bước 3: Tải các tệp dữ liệu nguồn (CSV) lên môi trường Colab

In [37]:
from google.colab import files

uploaded = files.upload()

print("Các file đã tải lên:", list(uploaded.keys()))

Saving customer.csv to customer.csv
Saving ecom_sales.csv to ecom_sales.csv
Saving product.csv to product.csv
Saving region.csv to region.csv
Các file đã tải lên: ['customer.csv', 'ecom_sales.csv', 'product.csv', 'region.csv']


### Bước 4: Khởi tạo và nạp dữ liệu vào tầng Bronze (Raw Data)
Đọc dữ liệu từ các tệp CSV vừa tải lên bằng Pandas và đẩy trực tiếp vào các bảng tương ứng trong Dataset Bronze trên BigQuery với chế độ tự động nhận diện schema.

In [38]:
source_files = {
    "ecom_sales": "ecom_sales.csv",
    "customer": "customer.csv",
    "product": "product.csv",
    "region": "region.csv"
}

DATASET_ID = f"{PROJECT_ID}.Dataset1_Bronze"

for table_name, file_name in source_files.items():
    if file_name not in uploaded:
        raise FileNotFoundError(
            f"Chưa tải lên file {file_name}"
        )

    df = pd.read_csv(file_name, encoding="utf-8-sig")

    table_id = f"{DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        write_disposition="WRITE_TRUNCATE"
    )

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    job.result()

    print(
        f"Đã nạp {table_name}: "
        f"{len(df):,} dòng → {table_id}"
    )

Đã nạp ecom_sales: 51,290 dòng → formidable-app-510501-p3.Dataset1_Bronze.ecom_sales
Đã nạp customer: 17,415 dòng → formidable-app-510501-p3.Dataset1_Bronze.customer
Đã nạp product: 28,356 dòng → formidable-app-510501-p3.Dataset1_Bronze.product
Đã nạp region: 3,828 dòng → formidable-app-510501-p3.Dataset1_Bronze.region


### Bước 5: Cấu trúc thư mục cho dự án DBT
Tạo cấu trúc thư mục tiêu chuẩn cho dự án dbt gồm thư mục cho tầng dữ liệu `silver` và `gold`.

In [39]:
import os

PROJECT_DIR = "/content/ecommerce_dbt"

os.makedirs(f"{PROJECT_DIR}/models/silver", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models/gold", exist_ok=True)

%cd /content/ecommerce_dbt

/content/ecommerce_dbt


### Bước 6: Khảo sát thông tin dữ liệu gốc tại tầng Bronze
Truy vấn thông tin cấu trúc schema, số lượng dòng của các bảng dữ liệu thô tại dataset Bronze để làm cơ sở xây dựng quy tắc làm sạch dữ liệu.

In [40]:

from google.cloud import bigquery

PROJECT_ID = "formidable-app-510501-p3"
BRONZE_DATASET = "Dataset1_Bronze"

client = bigquery.Client(project=PROJECT_ID)

bronze = client.get_dataset(
    f"{PROJECT_ID}.{BRONZE_DATASET}"
)

print("Bronze location:", bronze.location)
print("\nCác bảng Bronze:")

for table in client.list_tables(
    f"{PROJECT_ID}.{BRONZE_DATASET}"
):
    obj = client.get_table(table)
    print(f"\n{table.table_id}: {obj.num_rows:,} dòng")

    for field in obj.schema:
        print(f"  {field.name}: {field.field_type}")

Bronze location: US

Các bảng Bronze:

customer: 17,415 dòng
  customer_id: STRING
  first_name: STRING
  last_name: STRING
  birth_date: STRING
  marital_status: STRING
  gender: STRING
  email_address: STRING
  annual_income: INTEGER
  education_level: STRING
  occupation: STRING
  home_owner: STRING

ecom_sales: 51,290 dòng
  row_id: INTEGER
  order_id: STRING
  order_date: STRING
  customer_id: STRING
  segment: STRING
  region_code: STRING
  product_code: STRING
  quantity: INTEGER
  sales: FLOAT
  discount: FLOAT
  profit: FLOAT

product: 28,356 dòng
  product_code: STRING
  product: STRING
  category: STRING
  subcategory: STRING

region: 3,828 dòng
  region_code: STRING
  city: STRING
  state: STRING
  country: STRING
  country_latitude: FLOAT
  country_longitude: FLOAT
  region: STRING
  market: STRING


### Bước 7: Cấu hình dbt Profile và khởi tạo Dataset Silver
Kiểm tra vùng địa lý (Location) của dữ liệu để khởi tạo Dataset Silver cùng phân vùng. Đồng thời ghi đè file cấu hình kết nối `profiles.yml` cho dbt.

In [41]:

from pathlib import Path

SILVER_DATASET = "Dataset2_Silver"
bronze = client.get_dataset(
    f"{PROJECT_ID}.Dataset1_Bronze"
)

location = bronze.location

# Tạo dataset Silver nếu chưa tồn tại.
silver_ref = f"{PROJECT_ID}.{SILVER_DATASET}"

try:
    silver = client.get_dataset(silver_ref)
    if silver.location != location:
        raise ValueError(
            f"Silver đang ở {silver.location}, "
            f"nhưng Bronze ở {location}. "
            "Cần xử lý khác biệt location trước khi tiếp tục."
        )
    print("Dataset Silver đã tồn tại:", silver_ref)
except Exception as exc:
    if isinstance(exc, ValueError):
        raise
    from google.api_core.exceptions import NotFound

    if not isinstance(exc, NotFound):
        raise

    silver = bigquery.Dataset(silver_ref)
    silver.location = location
    client.create_dataset(silver)
    print("Đã tạo dataset Silver:", silver_ref)

profile_dir = Path.home() / ".dbt"
profile_dir.mkdir(parents=True, exist_ok=True)

profile = f"""ecommerce_dbt:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: "{PROJECT_ID}"
      dataset: "{SILVER_DATASET}"
      threads: 4
      location: "{location}"
      priority: interactive
"""

(profile_dir / "profiles.yml").write_text(
    profile, encoding="utf-8"
)

print("Đã cập nhật profile.")
print("Location:", location)

Dataset Silver đã tồn tại: formidable-app-510501-p3.Dataset2_Silver
Đã cập nhật profile.
Location: US


### Bước 8: Kiểm tra cấu hình và kết nối dbt
Chạy lệnh kiểm tra phiên bản và gỡ lỗi kết nối của dbt (`dbt debug`) để đảm bảo hệ thống có thể truy cập thông suốt đến BigQuery.

In [42]:

%cd /content/ecommerce_dbt
!dbt --version
!dbt debug --project-dir /content/ecommerce_dbt --profiles-dir /root/.dbt

/content/ecommerce_dbt
Core:
  - installed: 1.13.0-b1
  - latest:    1.12.6    - Ahead of latest version!

Plugins:
  - bigquery: 1.12.1 - Up to date!


00:31:40  dbt v2 is available, and is the default for new installations. This is dbt Core v1.
          New project - install v2: https://docs.getdbt.com/docs/local/install-dbt?utm_source=dbt-cli
          Existing v1 project - upgrade guide: https://docs.getdbt.com/docs/dbt-versions/dbt-upgrade/upgrading-to-v2?utm_source=dbt-cli
00:31:47  Running with dbt=1.13.0-b1
00:31:47  dbt version: 1.13.0-b1
00:31:47  python version: 3.13.16
00:31:47  python path: /usr/bin/python3
00:31:47  os info: Linux-6.6.122+-x86_64-with-glibc2.39
00:31:48  Using profiles dir at /root/.dbt
00:31:48  Using profiles.yml file at /root/.dbt/profiles.yml
00:31:48  Using dbt_project.yml file at /content/ecommerce_dbt/dbt_project.yml
00:31:48  adapter type: bigquery
00:31:48  adapter version: 1.12.1
00:31:49  Configuration:
00:31:49    profiles.yml file [OK found 

### Bước 9: Khai báo nguồn dữ liệu dbt (Sources YAML)
Tạo tệp `sources.yml` để định nghĩa tường minh các bảng nguồn nằm tại Dataset Bronze mà dbt sẽ sử dụng làm đầu vào.

In [43]:

from pathlib import Path

PROJECT_DIR = Path("/content/ecommerce_dbt")
SILVER_DIR = PROJECT_DIR / "models" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

sources_yaml = """version: 2

sources:
  - name: bronze
    database: formidable-app-510501-p3
    schema: Dataset1_Bronze
    tables:
      - name: ecom_sales
      - name: customer
      - name: product
      - name: region
"""

source_file = SILVER_DIR / "sources.yml"
source_file.write_text(sources_yaml, encoding="utf-8")

print("Đã tạo:", source_file)
print(source_file.read_text(encoding="utf-8"))

Đã tạo: /content/ecommerce_dbt/models/silver/sources.yml
version: 2

sources:
  - name: bronze
    database: formidable-app-510501-p3
    schema: Dataset1_Bronze
    tables:
      - name: ecom_sales
      - name: customer
      - name: product
      - name: region



### Bước 10: Thiết lập cấu hình vật lý hóa (Materialization) cho dbt
Cấu hình tệp `dbt_project.yml` để chuyển đổi phương thức lưu trữ của tầng Silver từ dạng View sang dạng bảng vật lý (`table`) giúp tăng tốc độ truy vấn.

In [44]:
# [Colab Cell Mới 1] Cập nhật dbt_project.yml sang materialization 'table'
from pathlib import Path

PROJECT_DIR = Path("/content/ecommerce_dbt")
config_path = PROJECT_DIR / "dbt_project.yml"

dbt_project_content = """name: 'ecommerce_dbt'
version: '1.0.0'
config-version: 2

profile: 'ecommerce_dbt'

model-paths: ['models']
analysis-paths: ['analyses']
test-paths: ['tests']
seed-paths: ['seeds']
macro-paths: ['macros']
snapshot-paths: ['snapshots']

clean-targets:
  - 'target'
  - 'dbt_packages'

models:
  ecommerce_dbt:
    silver:
      +materialized: table
"""

config_path.write_text(dbt_project_content, encoding="utf-8")
print("✅ Đã cấu hình Silver tạo thành TABLE trên BigQuery!")

✅ Đã cấu hình Silver tạo thành TABLE trên BigQuery!


### Bước 11: Xây dựng các mô hình biến đổi dữ liệu tầng Silver (Models)
Viết các tệp SQL logic để chuẩn hóa dữ liệu:
1. **silver_customer**: Chuẩn hóa định dạng, ghép họ tên, xử lý giá trị khuyết thiếu ở trường giới tính.
2. **silver_product**: Chuẩn hóa danh mục sản phẩm.
3. **silver_region**: Làm sạch dữ liệu địa lý vùng miền.
4. **silver_sales_integrated**: Tích hợp dữ liệu doanh thu đa chiều, kiểm tra tính hợp lệ của khóa ngoại.

In [45]:
# [Colab Cell Mới 2] Tạo các file Model SQL cho tầng Silver
from pathlib import Path

SILVER_DIR = Path("/content/ecommerce_dbt/models/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

models = {
    # 1. Làm sạch bảng Khách hàng (Xử lý 124 dòng gender NULL, ghép họ tên, tính tuổi)
    "silver_customer.sql": """
{{ config(materialized='table') }}

WITH raw_customer AS (
    SELECT * FROM {{ source('bronze', 'customer') }}
)
SELECT
    TRIM(customer_id) AS customer_id,
    TRIM(first_name) AS first_name,
    TRIM(last_name) AS last_name,
    CONCAT(TRIM(first_name), ' ', TRIM(last_name)) AS full_name,
    SAFE_CAST(birth_date AS DATE) AS birth_date,
    DATE_DIFF(CURRENT_DATE(), SAFE_CAST(birth_date AS DATE), YEAR) AS age,
    TRIM(marital_status) AS marital_status,
    -- Quy tắc Task 1: Xử lý 124 dòng thiếu giới tính thành 'Unknown'
    COALESCE(NULLIF(TRIM(gender), ''), 'Unknown') AS gender,
    TRIM(email_address) AS email_address,
    SAFE_CAST(annual_income AS INT64) AS annual_income,
    TRIM(education_level) AS education_level,
    TRIM(occupation) AS occupation,
    TRIM(home_owner) AS home_owner
FROM raw_customer
""",

    # 2. Làm sạch bảng Sản phẩm
    "silver_product.sql": """
{{ config(materialized='table') }}

WITH raw_product AS (
    SELECT * FROM {{ source('bronze', 'product') }}
)
SELECT
    TRIM(product_code) AS product_code,
    TRIM(product) AS product_name,
    TRIM(category) AS category,
    TRIM(subcategory) AS subcategory
FROM raw_product
""",

    # 3. Làm sạch bảng Khu vực
    "silver_region.sql": """
{{ config(materialized='table') }}

WITH raw_region AS (
    SELECT * FROM {{ source('bronze', 'region') }}
)
SELECT
    TRIM(region_code) AS region_code,
    TRIM(city) AS city,
    TRIM(state) AS state,
    TRIM(country) AS country,
    SAFE_CAST(country_latitude AS FLOAT64) AS country_latitude,
    SAFE_CAST(country_longitude AS FLOAT64) AS country_longitude,
    TRIM(region) AS region,
    TRIM(market) AS market
FROM raw_region
""",

    # 4. BẢNG TÍCH HỢP SILVER: Kết hợp đơn hàng với Customer, Product, Region
    # Giữ nguyên 51,290 dòng, đánh cờ và điền nhãn cho 3 dòng ngoại lệ phát hiện ở Task 1
    "silver_sales_integrated.sql": """
{{ config(materialized='table') }}

WITH raw_sales AS (
    SELECT * FROM {{ source('bronze', 'ecom_sales') }}
),
cust AS (
    SELECT * FROM {{ ref('silver_customer') }}
),
prod AS (
    SELECT * FROM {{ ref('silver_product') }}
),
reg AS (
    SELECT * FROM {{ ref('silver_region') }}
)

SELECT
    s.row_id,
    s.order_id,
    SAFE_CAST(s.order_date AS DATE) AS order_date,
    s.customer_id,
    c.full_name AS customer_name,
    c.gender AS customer_gender,
    c.age AS customer_age,

    s.product_code,
    -- Xử lý 2 dòng ngoại lệ product (PPP000010, PPP000002) không có trong danh mục
    COALESCE(p.product_name, 'Mã chưa xác định') AS product_name,
    COALESCE(p.category, 'Unknown') AS product_category,
    COALESCE(p.subcategory, 'Unknown') AS product_subcategory,
    CASE WHEN p.product_code IS NULL THEN FALSE ELSE TRUE END AS is_valid_product,

    s.region_code,
    -- Xử lý 2 dòng ngoại lệ region (RRR0001) không có trong danh mục
    COALESCE(r.city, 'Unknown') AS city,
    COALESCE(r.country, 'Unknown') AS country,
    COALESCE(r.region, 'Unknown') AS region,
    COALESCE(r.market, 'Unknown') AS market,
    CASE WHEN r.region_code IS NULL THEN FALSE ELSE TRUE END AS is_valid_region,

    s.segment,
    SAFE_CAST(s.quantity AS INT64) AS quantity,
    SAFE_CAST(s.sales AS FLOAT64) AS sales,
    SAFE_CAST(s.discount AS FLOAT64) AS discount,
    -- Giữ nguyên profit âm (11,041 dòng bán lỗ, đúng thực tế nghiệp vụ)
    SAFE_CAST(s.profit AS FLOAT64) AS profit

FROM raw_sales s
LEFT JOIN cust c ON s.customer_id = c.customer_id
LEFT JOIN prod p ON s.product_code = p.product_code
LEFT JOIN reg r ON s.region_code = r.region_code
"""
}

# Ghi đè lại các file model trong thư mục silver
for filename, content in models.items():
    file_path = SILVER_DIR / filename
    file_path.write_text(content.strip() + "\n", encoding="utf-8")
    print(f"✅ Đã tạo model: {filename}")

✅ Đã tạo model: silver_customer.sql
✅ Đã tạo model: silver_product.sql
✅ Đã tạo model: silver_region.sql
✅ Đã tạo model: silver_sales_integrated.sql


### Bước 12: Thiết lập kịch bản kiểm thử chất lượng dữ liệu (Data Tests)
Tạo tệp cấu hình `schema.yml` để tự động kiểm tra các ràng buộc quan trọng như: Tính duy nhất (unique), Không rỗng (not_null), giới hạn giá trị (accepted_values) và ràng buộc khóa ngoại (relationships).

In [46]:
# [Colab Cell Mới 3] Tạo file kiểm thử schema.yml
from pathlib import Path

schema_yml = """version: 2

models:
  - name: silver_customer
    description: "Bảng chiều Khách hàng đã làm sạch"
    columns:
      - name: customer_id
        tests:
          - unique
          - not_null
      - name: gender
        tests:
          - accepted_values:
              values: ['M', 'F', 'Unknown']

  - name: silver_product
    description: "Bảng chiều Sản phẩm đã làm sạch"
    columns:
      - name: product_code
        tests:
          - unique
          - not_null

  - name: silver_region
    description: "Bảng chiều Vùng miền đã làm sạch"
    columns:
      - name: region_code
        tests:
          - unique
          - not_null

  - name: silver_sales_integrated
    description: "Bảng Fact Đơn hàng tích hợp đa chiều tầng Silver"
    columns:
      - name: row_id
        tests:
          - unique
          - not_null
      - name: order_id
        tests:
          - not_null
      - name: sales
        tests:
          - not_null
      - name: customer_id
        tests:
          - not_null
          - relationships:
              to: ref('silver_customer')
              field: customer_id
"""

test_file = Path("/content/ecommerce_dbt/models/silver/schema.yml")
test_file.write_text(schema_yml, encoding="utf-8")
print("✅ Đã tạo file kiểm thử schema.yml!")

✅ Đã tạo file kiểm thử schema.yml!


### Bước 13: Thực thi quá trình biến đổi dữ liệu (Run dbt)
Chạy lệnh `dbt run` để dbt tự động biên dịch và thực thi tuần tự các câu lệnh SQL, tạo ra các bảng dữ liệu sạch hoàn chỉnh trên BigQuery.

In [47]:
# [Colab Cell Mới 4] Chạy dbt run để tạo các bảng Silver trên BigQuery
%cd /content/ecommerce_dbt
!dbt run --select silver --profiles-dir /root/.dbt

/content/ecommerce_dbt
00:31:56  Running with dbt=1.13.0-b1
00:31:59  Registered adapter: bigquery=1.12.1
00:32:00  Found 8 models, 13 data tests, 4 sources, 561 macros
00:32:00  
00:32:00  Concurrency: 4 threads (target='dev')
00:32:00  
00:32:00  1 of 8 START sql table model Dataset2_Silver.silver_customer ................... [RUN]
00:32:00  3 of 8 START sql table model Dataset2_Silver.silver_region ..................... [RUN]
00:32:00  4 of 8 START sql table model Dataset2_Silver.stg_customer ...................... [RUN]
00:32:00  2 of 8 START sql table model Dataset2_Silver.silver_product .................... [RUN]
00:32:03  3 of 8 OK created sql table model Dataset2_Silver.silver_region ................ [CREATE TABLE (3.8k rows, 297.6 KiB processed) in 2.24s]
00:32:03  5 of 8 START sql table model Dataset2_Silver.stg_ecom_sales .................... [RUN]
00:32:03  2 of 8 OK created sql table model Dataset2_Silver.silver_product ............... [CREATE TABLE (28.4k rows, 2.3 MiB pr

### Bước 14: Thực thi kiểm tra chất lượng tự động (Test dbt)
Chạy lệnh `dbt test` để thực hiện toàn bộ các kịch bản kiểm thử chất lượng dữ liệu đã định nghĩa ở Bước 12.

In [48]:
# [Colab Cell Mới 5] Chạy dbt test để kiểm tra chất lượng dữ liệu
%cd /content/ecommerce_dbt
!dbt test --select silver --profiles-dir /root/.dbt

/content/ecommerce_dbt
00:32:13  Running with dbt=1.13.0-b1
00:32:15  Registered adapter: bigquery=1.12.1
00:32:16  Found 8 models, 13 data tests, 4 sources, 561 macros
00:32:16  
00:32:16  Concurrency: 4 threads (target='dev')
00:32:16  
00:32:17  2 of 13 START test not_null_silver_customer_customer_id ........................ [RUN]
00:32:17  3 of 13 START test not_null_silver_product_product_code ........................ [RUN]
00:32:17  1 of 13 START test accepted_values_silver_customer_gender__M__F__Unknown ....... [RUN]
00:32:17  4 of 13 START test not_null_silver_region_region_code .......................... [RUN]
00:32:18  3 of 13 PASS not_null_silver_product_product_code .............................. [PASS in 0.99s]
00:32:18  5 of 13 START test not_null_silver_sales_integrated_customer_id ................ [RUN]
00:32:18  1 of 13 PASS accepted_values_silver_customer_gender__M__F__Unknown ............. [PASS in 1.09s]
00:32:18  6 of 13 START test not_null_silver_sales_integrated_

### Bước 15: Đối chiếu và tổng hợp kết quả tầng dữ liệu Silver
Truy vấn trực tiếp bảng kết quả cuối cùng để kiểm tra tổng số dòng dữ liệu, số lượng dòng không xác định giới tính, dòng sai lệch danh mục và các đơn hàng ghi nhận lợi nhuận âm.

In [49]:
# [Colab Cell Mới 6] Kiểm tra các bảng đã tạo trong Dataset2_Silver và đếm số dòng
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="formidable-app-510501-p3")

query = """
SELECT
    'silver_sales_integrated' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT row_id) AS distinct_row_id,
    COUNTIF(customer_gender = 'Unknown') AS unknown_gender_rows,
    COUNTIF(is_valid_product = FALSE) AS invalid_product_rows,
    COUNTIF(is_valid_region = FALSE) AS invalid_region_rows,
    COUNTIF(profit < 0) AS negative_profit_rows
FROM `formidable-app-510501-p3.Dataset2_Silver.silver_sales_integrated`
"""

df_result = client.query(query).to_dataframe()
print("📊 KẾT QUẢ ĐỐI CHIẾU TẦNG SILVER:")
display(df_result)

📊 KẾT QUẢ ĐỐI CHIẾU TẦNG SILVER:


,table_name,total_rows,distinct_row_id,unknown_gender_rows,invalid_product_rows,invalid_region_rows,negative_profit_rows
0,silver_sales_integrated,51290,51290,362,2,2,11041
